In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install sentence-transformers

<hr>

# Projektbeitrag (Beispiel)

#### Die Lernenden erstellen aus dem Zusammenwirken verschiedener Sprachmodelle einen möglichen Workflow. Dies kann grundsätzlich auch mit einem Sprachmodell geschehen, wenn im Vorfeld entsprechende Anfragen generiert wurden und das Sprachmodell hierfür geeignet ist.

## Ziel: Niederschwellige Darstellung von Retrieval Augmented Generation (RAG)

> Unter Retrieval-Augmented Generation (RAG) versteht man ein Softwaresystem, welches Information Retrieval mit einem Large Language Model kombiniert. Eine Abfrage, welche an das System gestellt wird, kann hierbei auf Informationen aus (externen) Informationsquellen, Datenbanken oder dem World Wide Web zugreifen statt nur auf die Trainingsdaten des Modells. Dies erhöht die Genauigkeit und Robustheit der generierten Inhalte, indem es die Modelle mit aktuellen und spezifischen Informationen versorgt. Typische Anwendungsfälle sind der Zugriff von Chatbots auf interne (Unternehmens-)Daten oder die Bereitstellung von Sachinformationen, die ausschließlich aus verlässlichen Quellen stammen sollen.
> Quelle: https://de.wikipedia.org/wiki/Retrieval-Augmented_Generation

### Voraussetzungen:

#### Die Lernenden müssen die Materialien zu Embeddings und dem Chatbot kennengelernt haben.

### Was du nach dieser Einheit kannst:
- Du kannst erklären, wie RAG die Grenzen eines Sprachmodells überwindet.
- Du kannst die drei Phasen eines RAG-Systems (**Indexierung**, **Retrieval**, **Generierung**) benennen und beschreiben.
- Du kannst das Notebook auf eigene Datenquellen anpassen.

<br>

<hr>

## Der RAG-Prozess im Überblick

Ein RAG-System arbeitet in **drei Phasen**, die du in diesem Notebook schrittweise umsetzt:

| Phase | Bezeichnung | Was passiert? |
|---|---|---|
| 1 | **Indexierung** | Die Datenquellen werden in Zahlen (Embeddings) umgewandelt und gespeichert. |
| 2 | **Retrieval** | Der Userprompt wird ebenfalls in Zahlen umgewandelt. Dann wird verglichen, welche Datenquelle am ähnlichsten ist. |
| 3 | **Generierung** | Das Sprachmodell erzeugt aus dem Userprompt und der gefundenen Quelle eine natürlichsprachige Antwort. |

---

In [ ]:
from sentence_transformers import SentenceTransformer # Import der Funktion pipeline des Moduls transformers
from transformers import pipeline # Import der Funktion pipeline des Moduls transformers

### Phase 1 – Indexierung
#### Festlegung der Datenquellen, deren Inhalt nicht in einem Sprachmodell enthalten ist.
#### In der Praxis stammen diese aus externen Quellen, wie z.B. PDF-Dokumenten oder Webseiten.

In [ ]:

datenquelle= ["Die KI-Schule hat werktags von 8-15 Uhr geöffnet.", "Das Kollegium besteht aus 80 Lehrkräften.","Sie erreichen uns unter folgender Nummer: 0911-123456.","Die Teilnahme an den Kursen ist kostenlos."]

> **Aufgabe 1:** Ergänze die `datenquelle`-Liste um mindestens drei eigene Informationen (z.B. über deine eigene Schule oder einen selbst gewählten Kontext). Führe danach das Notebook erneut aus und beobachte, wie sich die Ähnlichkeitswerte weiter unten verändern.

### Das Modell errechnet jetzt die Vektordarstellung (Embedding) einer jeden Datenquelle

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
quellenEmbeddings = model.encode(datenquelle)
quellenEmbeddings

### Phase 2 – Retrieval
#### Der Userprompt (die Anfrage des Nutzers) wird nun ebenfalls in ein Embedding umgewandelt, um ihn mit den Datenquellen vergleichbar zu machen.

In [ ]:
userprompt="Bitte teilen Sie mir mit, wie ich Sie kontaktieren kann."

In [ ]:
userpromptEmbedding=model.encode(userprompt) # Die nummerische Darstellung (Embedding) des Userprompts wird erstellt.
# userpromptEmbedding # Kommentar löschen, falls das Embedding angezeigt werden soll.

#### Die Ähnlichkeit (Kosinusähnlichkeit) zwischen dem Userprompt und jeder Datenquelle wird berechnet.
Der Wert liegt zwischen **-1** (völlig unähnlich) und **+1** (identisch). Werte über **0,5** deuten auf eine inhaltliche Übereinstimmung hin. Das System wählt automatisch die Datenquelle mit dem höchsten Wert aus.

In [ ]:
# Das Dictionary wird dynamisch erstellt
ergebnis_dict = {
    datenquelle[i]: model.similarity(userpromptEmbedding, quellenEmbeddings[i]).item()
    for i in range(len(quellenEmbeddings))
}

In [ ]:
for key, value in ergebnis_dict.items():
    print(f"{key} : Ähnlichkeit: {value}.")

> **Aufgabe 2:** Ändere den `userprompt` auf eine Frage, die **keine** passende Antwort in den Datenquellen hat (z.B. `"Wie lautet das Rezept für Apfelstrudel?"`). Führe das Notebook erneut aus. Was beobachtest du bei den Ähnlichkeitswerten? Welches grundsätzliche Problem entsteht für ein RAG-System, wenn keine passende Datenquelle vorhanden ist?

### Phase 3 – Generierung
#### Das Sprachmodell erzeugt jetzt eine natürlichsprachige Antwort aus dem Userprompt und der wahrscheinlichsten Datenquelle.

In [ ]:
max_key = max(ergebnis_dict, key=ergebnis_dict.get)
print("Wahrscheinlichste Datenquelle: " + max_key)
antwort = max_key

# device=-1 verwendet die CPU (funktioniert auf jedem Rechner).
# Falls eine NVIDIA-GPU vorhanden ist, kann device=0 für schnellere Verarbeitung genutzt werden.
# temperature=0.01 sorgt für sehr deterministische (wenig zufällige) Ausgaben.
# Höhere Werte (z.B. 0.8) machen die Antwort kreativer, aber weniger präzise.
pipe = pipeline("text-generation", model="LiquidAI/LFM2-350M", device=-1, max_new_tokens=500, temperature=0.01)

# Die # dienen als Trennzeichen im Prompt, damit das Modell Anfrage und Antwort klar unterscheiden kann (Prompt Engineering).
messages = [
    {"role": "system", "content": "Du bist ein nützlicher Chatbot und bekommst folgende Anfrage des Users: #"+userprompt+"#. Die Antwort auf die Frage lautet:#"+antwort+"#. Gib die Antwort auf die Frage in eigenen Worten passend zur Anfrage wieder. Fasse Dich kurz!"}
]
antwortprompt = pipe(messages)
antwortprompt[0]['generated_text'][-1]['content']

---
> **Aufgabe 3 – Reflexion:** Diskutiere folgende Fragen:
> - Was passiert, wenn zwei Datenquellen eine ähnlich hohe Ähnlichkeit aufweisen? Wie könnte man damit umgehen?
> - Wo liegt der Unterschied zwischen einem normalen Chatbot und einem RAG-System?
> - Welche Vor- und Nachteile siehst du bei diesem Ansatz im Vergleich zu einem Sprachmodell, das alle Informationen in seinen Trainingsdaten hat?